# Step 4 · Drive SAP → Source-aware Blueprint → Editable DOCX

**No SAP upload dialog.** This Notebook reads a PDF directly from your mounted Google Drive, checks the 431-template official Library, and saves the reconstructed Word + JSON reports back to Drive.

- **Default (SAP-only):** automatic extraction of **one Overall AE Table** (existing Step 2 pilot, two treatment arms).
- **Other Table families:** set `TARGET_SPEC_RELATIVE_PATH` to a **SAP-derived Table Specification JSON already in Google Drive**. The generic Step 4 rebuilds Demographics/Disposition-style layouts, but does **not** auto-extract their specifications yet.
- Each run writes a fresh folder, so earlier reports are not overwritten. No human approval is required to generate an **unapproved draft**.

The target study's TFL shell pages must not be supplied as extraction evidence. Generated Word output is a structural draft, not a clinically validated TFL.

## 0 · Bootstrap project and verify the active Colab Python kernel

In [ ]:
import importlib
import subprocess
import sys
from pathlib import Path

REPO_URL = "https://github.com/wohaofanaweishenme-cell/tfl-generation-agent.git"
GIT_REF = "feature/official-library-colab"
PROJECT_ROOT = Path("/content/tfl-generation-agent")
SOURCE_DIR = PROJECT_ROOT / "src"
PACKAGE_DIR = SOURCE_DIR / "tfl_agent"

def prepare_tfl_agent(*, update=True):
    """Install + import into THIS Colab kernel (works with a src/ layout)."""
    if PROJECT_ROOT.exists():
        if not (PROJECT_ROOT / ".git").is_dir():
            raise RuntimeError(
                f"{PROJECT_ROOT} exists but is not a git checkout; "
                "move this unrelated directory before re-running setup."
            )
        if update:
            subprocess.run(
                ["git", "-C", str(PROJECT_ROOT), "fetch", "origin", GIT_REF],
                check=True,
            )
            subprocess.run(
                ["git", "-C", str(PROJECT_ROOT), "checkout", "-B", GIT_REF, "FETCH_HEAD"],
                check=True,
            )
    else:
        subprocess.run(
            ["git", "clone", "--depth", "1", "--branch", GIT_REF,
             REPO_URL, str(PROJECT_ROOT)],
            check=True,
        )

    if not (PACKAGE_DIR / "__init__.py").is_file():
        raise FileNotFoundError(
            f"Source package missing: {PACKAGE_DIR}. Check git clone output."
        )
    # Editable pip install is helpful for dependencies, but a live notebook
    # kernel may not immediately discover a new editable-install .pth file.
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q", "-e", str(PROJECT_ROOT)],
        check=True,
    )
    if str(SOURCE_DIR) in sys.path:
        sys.path.remove(str(SOURCE_DIR))
    sys.path.insert(0, str(SOURCE_DIR))
    importlib.invalidate_caches()
    import tfl_agent
    imported_path = Path(tfl_agent.__file__).resolve()
    if not imported_path.is_relative_to(PACKAGE_DIR.resolve()):
        raise RuntimeError(
            "A different tfl_agent package is already cached in this kernel: "
            f"{imported_path}. Restart runtime and run all cells."
        )
    print("Active Python:", sys.executable)
    print("Project commit:", subprocess.check_output(
        ["git", "-C", str(PROJECT_ROOT), "rev-parse", "--short", "HEAD"],
        text=True,
    ).strip())
    print("Package imported successfully:", imported_path)
    return imported_path

prepare_tfl_agent()


## 1 · Mount Google Drive and choose SAP (no manual upload)

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

# Edit this path when you want to test a NEW SAP stored on your Google Drive.
# Path is relative to MyDrive, NOT to /content or the GitHub checkout.
DRIVE_SAP_RELATIVE_PATH = "Colab Notebooks/agent/midterm demo/SAP_001_00.pdf"

# Optional: a Table Specification JSON derived from THIS SAP and saved in Drive.
# Leave None for automatic Overall AE extraction from the SAP.
# Supply e.g. "Colab Notebooks/agent/new_study/14.1.2.1_spec.json" for other Tables.
TARGET_SPEC_RELATIVE_PATH = None

# OPTIONAL only when using an external Spec: authorized SAP body PDF page numbers.
# If unknown, leave None; the system will NOT search all pages (to prevent
# accidentally using held-out target TFL Shell pages as evidence).
SAP_BODY_PAGES = None  # e.g. (4, 21)

MYDRIVE = Path("/content/drive/MyDrive")
SAP_PATH = (MYDRIVE / DRIVE_SAP_RELATIVE_PATH).resolve()
if not SAP_PATH.is_file() or SAP_PATH.suffix.lower() != ".pdf":
    raise FileNotFoundError(f"SAP PDF missing in Drive: {SAP_PATH}")
SPEC_PATH = ((MYDRIVE / TARGET_SPEC_RELATIVE_PATH).resolve()
             if TARGET_SPEC_RELATIVE_PATH else None)
if SPEC_PATH is not None and not SPEC_PATH.is_file():
    raise FileNotFoundError(f"Target Specification JSON missing in Drive: {SPEC_PATH}")
print("SAP:", SAP_PATH)
print("Specification mode:", "automatic AE" if SPEC_PATH is None else str(SPEC_PATH))


## 2 · Run source-aware reconstruction and export to Google Drive

In [ ]:
# Self-healing import guard: safe even when Step 0 was skipped in this runtime.
# SAP is still read from Google Drive, never uploaded or copied into the repo.
import importlib
import subprocess
import sys
from pathlib import Path

PROJECT_ROOT = Path("/content/tfl-generation-agent")
SOURCE_DIR = PROJECT_ROOT / "src"
PACKAGE_DIR = SOURCE_DIR / "tfl_agent"
if not (PACKAGE_DIR / "__init__.py").is_file():
    # Execute Step 0's bootstrap in this kernel if this cell was run directly.
    if not PROJECT_ROOT.exists():
        subprocess.run(
            ["git", "clone", "--depth", "1",
             "--branch", "feature/official-library-colab",
             "https://github.com/wohaofanaweishenme-cell/tfl-generation-agent.git",
             str(PROJECT_ROOT)], check=True,
        )
    if not (PACKAGE_DIR / "__init__.py").is_file():
        raise FileNotFoundError(
            f"Missing {PACKAGE_DIR}. Run Step 0 and check the clone log."
        )
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q", "-e", str(PROJECT_ROOT)],
        check=True,
    )
if str(SOURCE_DIR) not in sys.path:
    sys.path.insert(0, str(SOURCE_DIR))
importlib.invalidate_caches()
import tfl_agent
if not Path(tfl_agent.__file__).resolve().is_relative_to(PACKAGE_DIR.resolve()):
    raise RuntimeError(
        "Incorrect cached tfl_agent package. Restart runtime, then Run all."
    )
print("Using local tfl_agent:", tfl_agent.__file__)

import json
import shutil
from datetime import datetime, timezone
from tfl_agent.agent.end_to_end_ae_pilot import run_single_sap_ae_pilot
from tfl_agent.agent.table_blueprint_pipeline import run_table_blueprint_from_spec

RUN_ID = datetime.now(timezone.utc).strftime("%Y%m%d_%H%M%S_%f")
OUTPUT_DIR = SAP_PATH.parent / "Blueprint_Reconstruction_Outputs" / f"{SAP_PATH.stem}_{RUN_ID}"
OUTPUT_DIR.mkdir(parents=True, exist_ok=False)

if SPEC_PATH is None:
    result = run_single_sap_ae_pilot(
        sap_path=SAP_PATH,
        library_path=PROJECT_ROOT / "library",
        output_dir=OUTPUT_DIR,
    )
else:
    result = run_table_blueprint_from_spec(
        sap_path=SAP_PATH,
        spec_path=SPEC_PATH,
        library_path=PROJECT_ROOT / "library",
        output_dir=OUTPUT_DIR,
        sap_body_page_range=SAP_BODY_PAGES,
    )
print(json.dumps({
    "status": result["status"],
    "target_id": result["target_id"],
    "approved_library_count": result["approved_library_count"],
    "source_structure_used": result["source_structure_used"],
    "validation": result["validation"]["valid_structural_shell"],
}, indent=2, ensure_ascii=False))
print("Word and JSON files:", OUTPUT_DIR)


## 3 · Inspect Blueprint and pack results (also saved in Drive)

In [ ]:
from IPython.display import display, JSON
blueprint = json.loads((OUTPUT_DIR / "Reconstruction_Blueprint.json").read_text(encoding="utf-8"))
print("Source structure knowledge used:")
display(JSON(blueprint["source_structure"]))
print("Word columns:", [blueprint["row_label_header"]] +
      [c["label"] for c in blueprint["columns"]])
print("Word body rows:", len(blueprint["rows"]))
print("Unresolved:", blueprint["unresolved_requirements"])
print("\nGenerated files:")
for p in sorted(OUTPUT_DIR.iterdir()):
    print(" -", p.name)
ZIP_FILE = Path(shutil.make_archive(str(OUTPUT_DIR), "zip", root_dir=OUTPUT_DIR))
print("\nGoogle Drive ZIP:", ZIP_FILE)


## Reading the results

- `Reconstruction_Blueprint.json`: the exact row/column hierarchy the Word generator consumed.
- `Adaptation_Plan.json` (when available): diagnostic Source IR / Target IR / differences.
- `Generation_Rationale.json` and `Validation_Report.json`: what was generated and structurally checked.
- `*.docx`: an editable **unapproved structural draft**.

**Important limitation:** the default SAP-only Step 2 still supports a recognizable Overall AE section and two explicit treatment arms. For a different type of TFL, prepare a target SAP-derived Specification JSON first; expanding Step 2 across all Table families remains separate work.